# J2 Après-Midi | L'API Hour : Collecte de Données Web

**Instructeur :** Mickael Temporão

**Durée estimée :** 1h15

**Objectifs d'apprentissage :**
1. Comprendre le fonctionnement d'une requête HTTP et le concept d'API.
2. Interroger des API existantes (ex: API Wikipédia) pour extraire des données structurées.
3. Comprendre la structure d'une page HTML.
4. Réaliser un premier script de *Web Scraping* avec BeautifulSoup pour collecter des métadonnées (ex: INA).

**Prérequis :**
- Notions basiques de Python (J1)
- Compte Google (pour exécuter ce notebook sur Colab)

**Packages requis :** `requests`, `pandas`, `mwclient`, `beautifulsoup4`

In [ ]:
!uv pip install --system requests pandas mwclient beautifulsoup4 lxml

## 1. Qu'est-ce qu'une API ? (Concept et Pratique)

En sciences sociales, on entend souvent parler d'API pour extraire des données de Twitter/X, Wikipedia ou des bases de données institutionnelles.

**API** signifie *Application Programming Interface* (Interface de Programmation d'Application).
L'analogie la plus connue est celle du **restaurant** :
- Vous (l'utilisateur ou votre code Python) êtes le client à la table.
- La cuisine (le serveur web ou la base de données) prépare les plats (les données).
- **L'API, c'est le serveur (waiter)** : vous lui donnez votre commande (la *requête* ou *request*), il la transmet à la cuisine, et il vous rapporte votre plat (la *réponse*, généralement en format `JSON`).

Faisons une requête HTTP basique pour demander à une API publique (l'API de recherche d'OpenData) de nous renvoyer une information.

In [ ]:
import requests
import json

# L'adresse à laquelle on passe notre commande (l'URL de l'API)
url = "https://data.enseignementsup-recherche.gouv.fr/api/explore/v2.1/catalog/datasets"

# On définit nos paramètres (notre commande spécifique)
params = {
    "limit": 1  # On demande un seul résultat
}

# Le serveur HTTP apporte notre commande à la cuisine (via un GET)
response = requests.get(url, params=params)

# On vérifie si la commande a bien été reçue (Code 200 = OK)
print(f"Statut de la réponse : {response.status_code}")

# On affiche le 'plat' qu'il nous a ramené, formaté en JSON
data = response.json()
print("\nDonnées reçues (JSON) :")
print(json.dumps(data, indent=2, ensure_ascii=False)[:300] + "\n... [contenu tronqué]")

## 2. API Wikipédia : Extraire l'histoire d'une page

Pour un politiste ou un sociologue, les historiques de modification Wikipédia sont une mine d'or pour observer les controverses (ex: *edit wars*). 
Plutôt que de scrapper manuellement, nous allons utiliser l'API officielle de Wikipédia via une librairie Python simplifiée : `mwclient`.

In [ ]:
import mwclient
import difflib

# On se connecte à la version française de Wikipedia
site = mwclient.Site('fr.wikipedia.org')

def fetch_wikipedia_revisions(page_title, limit=5):
    """
    Récupère les dernières révisions d'une page Wikipédia.
    """
    page = site.pages[page_title]
    revisions = []
    
    # On demande à l'API de nous renvoyer le contenu et la date
    for rev in page.revisions(prop='content|timestamp', limit=limit):
        revisions.append(rev)
        
    return revisions

# Exemple avec une personnalité politique
page_title = 'Gabriel_Attal'
print(f"Extraction des révisions pour : {page_title}...")
revisions = fetch_wikipedia_revisions(page_title, limit=3)
print(f"{len(revisions)} révisions récupérées.")

### Hack-Time 🛠️

Modifiez le code ci-dessous pour : 
1. Choisir une autre page Wikipédia (ex: `Loi_immigration_en_France` ou le nom d'un autre politique).
2. Demander les 10 dernières révisions.
3. Afficher la date (`timestamp`) de chaque révision extraite. *(Astuce : dans la boucle, `rev['timestamp']` contient la date).*

**Défi optionnel** : Affichez la taille (`len(rev['*'])`) du texte de chaque révision.

In [ ]:
# À vous de jouer !
mon_titre = 'Gilets_jaunes_(mouvement)'
mes_revisions = fetch_wikipedia_revisions(mon_titre, limit=5)

for i, rev in enumerate(mes_revisions):
    # Imprimez la date de la révision ici
    pass 


## 3. Web Scraping : Extraire quand l'API n'existe pas (BeautifulSoup)

Que faire lorsqu'un site web ne propose pas d'API ? Il faut aller chercher l'information directement dans le code source de la page (HTML). C'est le **Web Scraping**.

Imaginons que nous voulions étudier le temps d'antenne consacré aux "Zones à Faibles Émissions" (ZFE) sur différentes chaînes d'information (TF1, CNews). Le catalogue de l'INA (Institut National de l'Audiovisuel) propose un moteur de recherche, mais pas d'API ouverte.

Nous allons interroger la page web et utiliser `BeautifulSoup` pour lire la structure HTML et extraire les données.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd

# URL du catalogue de l'INA
base_url = "https://catalogue.ina.fr/docListe/TV-RADIO"

# On reproduit les filtres de la barre de recherche
params_ina = {
    "base_label": "TVNAT,TVSAT",  # TV Nationale et Satellite
    "itoustext": "ZFE",           # Notre mot clé
    "ch": "CNews",                # La chaîne
    "nbLignes": 10                # Nombre de résultats par page
}

# On simule un navigateur standard
headers = {"User-Agent": "Mozilla/5.0"}

# 1. Récupération du code HTML
print("Téléchargement de la page de l'INA...")
response_ina = requests.get(base_url, params=params_ina, headers=headers)
html_content = response_ina.text

# 2. Parsing (Analyse de la structure) avec BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

# Dans le code HTML de l'INA, les résultats sont dans des lignes de tableau <tr> 
# ayant une classe qui contient 'result_line'.
rows = soup.find_all("tr", class_=lambda x: x and "result_line" in x)
print(f"Nombre de résultats trouvés sur cette page : {len(rows)}")

Dans la boucle suivante, nous allons inspecter chaque ligne (`row`) et extraire le texte des colonnes (`td`).

In [ ]:
ina_data = []

for row in rows:
    cells = row.find_all("td")
    if len(cells) >= 9:
        # L'indexation commence à 0. Dans ce tableau:
        # cells[1] = Chaîne, cells[2] = Date, cells[4] = Durée, cells[5] = Titre
        
        chaine = cells[1].get_text(strip=True)
        date = cells[2].get_text(strip=True)
        titre = cells[5].get_text(strip=True)
        
        ina_data.append({
            "chaine": chaine,
            "date": date,
            "titre": titre
        })

# On transforme notre liste en un tableau propre (DataFrame pandas)
df_ina = pd.DataFrame(ina_data)
display(df_ina.head())

### Hack-Time 🛠️

Observez comment le `titre` et la `date` ont été extraits en utilisant l'indice des cellules HTML (`cells[1]`, `cells[2]`, etc.).

Sachant que la colonne **"Durée"** se trouve à la 5ème place dans le tableau HTML de l'INA, modifiez le code ci-dessus pour ajouter également la durée de l'émission dans votre `DataFrame`.

## 4. Conclusion & Passage à l'échelle

Félicitations ! Vous venez d'extraire des données brutes depuis Wikipédia et le catalogue INA.

Dans un véritable projet de recherche, il faudrait exécuter cette boucle sur des centaines de pages pour construire notre corpus. **Cependant**, faire trop de requêtes web à la suite peut surcharger les serveurs de l'INA (qui pourraient bloquer notre adresse IP !).

Pour la suite de l'atelier, nous allons donc utiliser **un jeu de données pré-collecté** qui contient plusieurs centaines de résultats INA sur les "ZFE" entre TF1 et CNews.

*Rendez-vous dans le notebook de Prompt Engineering pour la suite !*